# Notebook 01 — Data Collection

## Purpose
Pull and clean all raw data for the project — all other notebooks read from `/data/processed/` rather than calling APIs directly.

## Data Sources
- **BoE portal** — UK nominal gilt yields (2yr, 5yr, 10yr, 30yr) and index-linked gilt yields
- **FRED API** — US Treasury yields, TIPS real yields, German Bund, BoE Bank Rate, Fed Funds Rate, GBPUSD, VIX
- **yfinance** — ETF price series: IGLT.L, INXG.L, TLT, GBPUSD=X
- **ONS** — UK CPI/RPI (manual download)

## Outputs
Cleaned `.csv` files saved to `/data/processed/` for use in all subsequent notebooks.

## Period
2016-01-01 to present (daily where available, monthly otherwise)

In [1]:
# ============================================================
# IMPORTS AND SETUP
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from dotenv import load_dotenv
from fredapi import Fred
import yfinance as yf
import warnings
warnings.filterwarnings('ignore')

# Load FRED API key from .env file
load_dotenv()
fred = Fred(api_key=os.getenv('FRED_API_KEY'))

# Date range for the full project
START_DATE = '2016-01-01'
END_DATE   = pd.Timestamp.today().strftime('%Y-%m-%d')

print(f"Setup complete. Data range: {START_DATE} to {END_DATE}")

Setup complete. Data range: 2016-01-01 to 2026-04-06


In [2]:
# ============================================================
# FRED DATA PULL — US TREASURIES, GERMAN BUND, MACRO VARIABLES
# ============================================================

# US Treasury yields
print("Pulling US Treasury yields...")
us_2yr  = fred.get_series('DGS2',  observation_start=START_DATE, observation_end=END_DATE)
us_5yr  = fred.get_series('DGS5',  observation_start=START_DATE, observation_end=END_DATE)
us_10yr = fred.get_series('DGS10', observation_start=START_DATE, observation_end=END_DATE)
us_30yr = fred.get_series('DGS30', observation_start=START_DATE, observation_end=END_DATE)

# US TIPS real yields and breakeven inflation
print("Pulling US TIPS and breakeven inflation...")
us_tips_5yr  = fred.get_series('DFII5',  observation_start=START_DATE, observation_end=END_DATE)
us_tips_10yr = fred.get_series('DFII10', observation_start=START_DATE, observation_end=END_DATE)
us_breakeven_10yr = fred.get_series('T10YIE', observation_start=START_DATE, observation_end=END_DATE)
us_5y5y_forward   = fred.get_series('T5YIFR', observation_start=START_DATE, observation_end=END_DATE)

# German Bund
print("Pulling German Bund...")
de_10yr = fred.get_series('IRLTLT01DEM156N', observation_start=START_DATE, observation_end=END_DATE)

# Central bank policy rates
print("Pulling policy rates...")
boe_rate  = fred.get_series('BOERUKM',  observation_start=START_DATE, observation_end=END_DATE)
fed_funds = fred.get_series('FEDFUNDS', observation_start=START_DATE, observation_end=END_DATE)
ecb_rate  = fred.get_series('ECBDFR',   observation_start=START_DATE, observation_end=END_DATE)

# Macro variables
print("Pulling macro variables...")
vix    = fred.get_series('VIXCLS',  observation_start=START_DATE, observation_end=END_DATE)
gbpusd = fred.get_series('DEXUSUK', observation_start=START_DATE, observation_end=END_DATE)

print("\nAll FRED series pulled successfully.")

Pulling US Treasury yields...
Pulling US TIPS and breakeven inflation...
Pulling German Bund...
Pulling policy rates...
Pulling macro variables...

All FRED series pulled successfully.


In [3]:
# ============================================================
# COMBINE FRED DATA INTO DATAFRAMES
# ============================================================

# US Treasuries
us_treasuries = pd.DataFrame({
    'us_2yr':  us_2yr,
    'us_5yr':  us_5yr,
    'us_10yr': us_10yr,
    'us_30yr': us_30yr,
})

# US TIPS and inflation expectations
us_tips = pd.DataFrame({
    'us_tips_5yr':        us_tips_5yr,
    'us_tips_10yr':       us_tips_10yr,
    'us_breakeven_10yr':  us_breakeven_10yr,
    'us_5y5y_forward':    us_5y5y_forward,
})

# German Bund
german_bund = pd.DataFrame({
    'de_10yr': de_10yr,
})

# Policy rates
policy_rates = pd.DataFrame({
    'boe_rate':  boe_rate,
    'fed_funds': fed_funds,
    'ecb_rate':  ecb_rate,
})

# Macro variables
macro_variables = pd.DataFrame({
    'vix':    vix,
    'gbpusd': gbpusd,
})

# Quick sense check — print shape and date range of each
for name, df in [('US Treasuries', us_treasuries), 
                 ('US TIPS', us_tips),
                 ('German Bund', german_bund),
                 ('Policy Rates', policy_rates),
                 ('Macro Variables', macro_variables)]:
    print(f"{name}: {df.shape[0]} rows, {df.index[0].date()} to {df.index[-1].date()}, "
          f"nulls: {df.isnull().sum().sum()}")

US Treasuries: 2675 rows, 2016-01-01 to 2026-04-02, nulls: 452
US TIPS: 2676 rows, 2016-01-01 to 2026-04-03, nulls: 454
German Bund: 121 rows, 2016-01-01 to 2026-01-01, nulls: 0
Policy Rates: 3745 rows, 2016-01-01 to 2026-04-02, nulls: 7354
Macro Variables: 2674 rows, 2016-01-01 to 2026-04-01, nulls: 187


In [4]:
# ============================================================
# INVESTIGATE POLICY RATES
# ============================================================

# Check what frequency each series actually is
print("BoE Rate — first 10 rows:")
print(boe_rate.head(10))
print(f"\nBoE Rate total rows: {len(boe_rate)}")

print("\nFed Funds — first 10 rows:")
print(fed_funds.head(10))
print(f"\nFed Funds total rows: {len(fed_funds)}")

print("\nECB Rate — first 10 rows:")
print(ecb_rate.head(10))
print(f"\nECB Rate total rows: {len(ecb_rate)}")

BoE Rate — first 10 rows:
2016-01-01    0.50
2016-02-01    0.50
2016-03-01    0.50
2016-04-01    0.50
2016-05-01    0.50
2016-06-01    0.50
2016-07-01    0.50
2016-08-01    0.25
2016-09-01    0.25
2016-10-01    0.25
dtype: float64

BoE Rate total rows: 13

Fed Funds — first 10 rows:
2016-01-01    0.34
2016-02-01    0.38
2016-03-01    0.36
2016-04-01    0.37
2016-05-01    0.37
2016-06-01    0.38
2016-07-01    0.39
2016-08-01    0.40
2016-09-01    0.40
2016-10-01    0.40
dtype: float64

Fed Funds total rows: 123

ECB Rate — first 10 rows:
2016-01-01   -0.3
2016-01-02   -0.3
2016-01-03   -0.3
2016-01-04   -0.3
2016-01-05   -0.3
2016-01-06   -0.3
2016-01-07   -0.3
2016-01-08   -0.3
2016-01-09   -0.3
2016-01-10   -0.3
dtype: float64

ECB Rate total rows: 3745


In [5]:
# ============================================================
# FIX POLICY RATES — KEEP SEPARATE, FIX BOE SERIES
# ============================================================

# BoE rate — BOERUKM is too sparse
# Use UKBRBASE which gives the full monthly series
print("Re-pulling BoE rate with correct series...")
boe_rate = fred.get_series('UKBRBASE', observation_start=START_DATE, observation_end=END_DATE)
print(f"BoE Rate (corrected): {len(boe_rate)} rows")
print(boe_rate.head(10))

Re-pulling BoE rate with correct series...


ValueError: Bad Request.  The series does not exist.

In [6]:
# ============================================================
# FIX BOE RATE — TRY ALTERNATIVE FRED SERIES
# ============================================================

# Try INTEREST RATE series for UK
candidates = ['IRSTCI01GBM156N', 'INTDSRGBM193N']

for code in candidates:
    try:
        test = fred.get_series(code, observation_start=START_DATE, observation_end=END_DATE)
        print(f"{code}: {len(test)} rows, first value: {test.iloc[0]}, last value: {test.iloc[-1]}")
    except Exception as e:
        print(f"{code}: FAILED — {e}")

IRSTCI01GBM156N: 122 rows, first value: 0.4659, last value: 3.7274
INTDSRGBM193N: FAILED — single positional indexer is out-of-bounds


In [7]:
# ============================================================
# STORE POLICY RATES CORRECTLY — ALL AS SEPARATE MONTHLY SERIES
# ============================================================

# BoE rate proxy — OECD 3-month interbank, monthly
boe_rate = fred.get_series('IRSTCI01GBM156N', 
                            observation_start=START_DATE, 
                            observation_end=END_DATE)

# Fed Funds and ECB already pulled correctly — just confirm
print("Policy rates summary:")
print(f"  BoE proxy (IRSTCI01GBM156N): {len(boe_rate)} rows, "
      f"{boe_rate.index[0].date()} to {boe_rate.index[-1].date()}")
print(f"  Fed Funds:                   {len(fed_funds)} rows, "
      f"{fed_funds.index[0].date()} to {fed_funds.index[-1].date()}")
print(f"  ECB Rate:                    {len(ecb_rate)} rows, "
      f"{ecb_rate.index[0].date()} to {ecb_rate.index[-1].date()}")

# Note for the writeup — document this data decision
print("\nNote: BoE series is OECD 3-month interbank rate (IRSTCI01GBM156N),")
print("not official Bank Rate. Tracks Bank Rate closely. Adequate for")
print("policy rate context and regression analysis.")

Policy rates summary:
  BoE proxy (IRSTCI01GBM156N): 122 rows, 2016-01-01 to 2026-02-01
  Fed Funds:                   123 rows, 2016-01-01 to 2026-03-01
  ECB Rate:                    3745 rows, 2016-01-01 to 2026-04-02

Note: BoE series is OECD 3-month interbank rate (IRSTCI01GBM156N),
not official Bank Rate. Tracks Bank Rate closely. Adequate for
policy rate context and regression analysis.


In [8]:
# ============================================================
# YFINANCE PULL — ETFs AND GBPUSD
# ============================================================

print("Pulling ETF and FX data from yfinance...")

tickers = {
    'IGLT.L': 'uk_gilt_etf',        # iShares UK Gilts ETF
    'INXG.L': 'uk_indexlinked_etf', # iShares UK Index-Linked Gilts ETF
    'TLT':    'us_treasury_etf',    # iShares US 20yr Treasury ETF
    'GLD':    'gold',               # Gold — safe haven context
    'GBPUSD=X': 'gbpusd_yf',        # Sterling vs USD
}

yf_data = {}
for ticker, name in tickers.items():
    try:
        df = yf.download(ticker, start=START_DATE, end=END_DATE, 
                         auto_adjust=True, progress=False)
        yf_data[name] = df['Close']
        print(f"  {ticker} ({name}): {len(df)} rows, "
              f"{df.index[0].date()} to {df.index[-1].date()}")
    except Exception as e:
        print(f"  {ticker}: FAILED — {e}")

print("\nAll yfinance tickers attempted.")

Pulling ETF and FX data from yfinance...
  IGLT.L (uk_gilt_etf): 2590 rows, 2016-01-04 to 2026-04-02
  INXG.L (uk_indexlinked_etf): 2590 rows, 2016-01-04 to 2026-04-02
  TLT (us_treasury_etf): 2577 rows, 2016-01-04 to 2026-04-02
  GLD (gold): 2577 rows, 2016-01-04 to 2026-04-02
  GBPUSD=X (gbpusd_yf): 2668 rows, 2016-01-01 to 2026-04-03

All yfinance tickers attempted.


In [9]:
# ============================================================
# BOE NOMINAL GILT SPOT CURVE — IMPORT AND PARSE
# ============================================================

import glob

# Target maturities we want to extract
TARGET_MATURITIES = [2.0, 5.0, 10.0, 30.0]

def parse_boe_spot_curve(filepath, sheet_name='spot curve', target_maturities=TARGET_MATURITIES):
    """
    Parse a BoE gilt spot curve Excel file.
    
    Structure:
    - Row 0: title (skip)
    - Row 1: blank (skip)  
    - Row 2: 'Maturity' label (skip)
    - Row 3: maturity years — use as column headers
    - Row 4: blank (skip)
    - Row 5+: data rows — date in col 0, yields in subsequent cols
    """
    
    # Read the raw Excel file with no header
    raw = pd.read_excel(filepath, sheet_name=sheet_name, header=None)
    
    # Extract maturity years from row 3 (index 3)
    maturity_row = raw.iloc[3, 1:].values.astype(float)
    
    # Find column indices for our target maturities
    col_indices = []
    col_names = []
    for mat in target_maturities:
        # Find closest match in maturity row
        idx = np.argmin(np.abs(maturity_row - mat))
        actual_mat = maturity_row[idx]
        col_indices.append(idx + 1)  # +1 because col 0 is the date
        col_names.append(f'uk_{int(mat)}yr')
        print(f"  Target {mat}yr — found column {actual_mat}yr at index {idx+1}")
    
    # Extract data rows — skip first 5 rows (rows 0-4)
    data = raw.iloc[5:, :]
    
    # Extract date column and target yield columns
    dates = pd.to_datetime(data.iloc[:, 0], dayfirst=True, errors='coerce')
    yields = data.iloc[:, col_indices].astype(float)
    
    # Build clean DataFrame
    df = pd.DataFrame(yields.values, index=dates, columns=col_names)
    
    # Drop rows where date failed to parse (blank rows etc)
    df = df[df.index.notna()]
    
    # Drop rows where all yields are NaN
    df = df.dropna(how='all')
    
    return df

# Process both files
print("Parsing 2016-2024 nominal gilt file...")
file_2016 = parse_boe_spot_curve('data/raw/GLC Nominal daily data_2016 to 2024.xlsx')
print(f"  Shape: {file_2016.shape}, {file_2016.index[0].date()} to {file_2016.index[-1].date()}")

print("\nParsing 2025-present nominal gilt file...")
file_2025 = parse_boe_spot_curve('data/raw/GLC Nominal daily data_2025 to present.xlsx')
print(f"  Shape: {file_2025.shape}, {file_2025.index[0].date()} to {file_2025.index[-1].date()}")

Parsing 2016-2024 nominal gilt file...


FileNotFoundError: [Errno 2] No such file or directory: 'data/raw/GLC Nominal daily data_2016 to 2024.xlsx'

In [10]:
import os

# Check what's actually in data/raw/
print("Files in data/raw/:")
for f in os.listdir('data/raw/'):
    print(f" {f}")

Files in data/raw/:


FileNotFoundError: [WinError 3] The system cannot find the path specified: 'data/raw/'

In [11]:
import os

# Find where Jupyter is currently running from
print("Current working directory:")
print(os.getcwd())


Current working directory:
C:\Users\tomsu\Documents\Graduate Jobs\CV PROJECTS\uk-gilt-market-analysis\notebooks


In [12]:
# Move up one level to project root
os.chdir('..')

# Confirm
print("Working directory now set to:")
print(os.getcwd())

# Check data/raw/ contents
print("\nFiles in data/raw/:")
for f in os.listdir('data/raw/'):
    print(f"  {f}")

Working directory now set to:
C:\Users\tomsu\Documents\Graduate Jobs\CV PROJECTS\uk-gilt-market-analysis

Files in data/raw/:
  GLC Nominal daily data_2016 to 2024.xlsx
  GLC Nominal daily data_2025 to present.xlsx


In [13]:
print("Parsing 2016-2024 nominal gilt file...")
file_2016 = parse_boe_spot_curve('data/raw/GLC Nominal daily data_2016 to 2024.xlsx')
print(f"  Shape: {file_2016.shape}, {file_2016.index[0].date()} to {file_2016.index[-1].date()}")

print("\nParsing 2025-present nominal gilt file...")
file_2025 = parse_boe_spot_curve('data/raw/GLC Nominal daily data_2025 to present.xlsx')
print(f"  Shape: {file_2025.shape}, {file_2025.index[0].date()} to {file_2025.index[-1].date()}")

Parsing 2016-2024 nominal gilt file...


ValueError: Worksheet named 'spot curve' not found

In [14]:
import openpyxl

for filepath in ['data/raw/GLC Nominal daily data_2016 to 2024.xlsx',
                 'data/raw/GLC Nominal daily data_2025 to present.xlsx']:
    wb = openpyxl.load_workbook(filepath, read_only=True)
    print(f"{filepath}:")
    for name in wb.sheetnames:
        print(f"  '{name}'")
    wb.close()

data/raw/GLC Nominal daily data_2016 to 2024.xlsx:
  'info'
  '1. fwds, short end'
  '2. fwd curve'
  '3. spot, short end'
  '4. spot curve'
data/raw/GLC Nominal daily data_2025 to present.xlsx:
  'info'
  '1. fwds, short end'
  '2. fwd curve'
  '3. spot, short end'
  '4. spot curve'


In [15]:
print("Parsing 2016-2024 nominal gilt file...")
file_2016 = parse_boe_spot_curve('data/raw/GLC Nominal daily data_2016 to 2024.xlsx',
                                  sheet_name='4. spot curve')
print(f"  Shape: {file_2016.shape}, {file_2016.index[0].date()} to {file_2016.index[-1].date()}")

print("\nParsing 2025-present nominal gilt file...")
file_2025 = parse_boe_spot_curve('data/raw/GLC Nominal daily data_2025 to present.xlsx',
                                  sheet_name='4. spot curve')
print(f"  Shape: {file_2025.shape}, {file_2025.index[0].date()} to {file_2025.index[-1].date()}")

Parsing 2016-2024 nominal gilt file...
  Target 2.0yr — found column 2.0yr at index 4
  Target 5.0yr — found column 5.0yr at index 10
  Target 10.0yr — found column 10.0yr at index 20
  Target 30.0yr — found column 30.0yr at index 60
  Shape: (2273, 4), 2016-01-04 to 2024-12-31

Parsing 2025-present nominal gilt file...
  Target 2.0yr — found column 2.0yr at index 4
  Target 5.0yr — found column 5.0yr at index 10
  Target 10.0yr — found column 10.0yr at index 20
  Target 30.0yr — found column 30.0yr at index 60
  Shape: (316, 4), 2025-01-02 to 2026-03-31


In [16]:
# ============================================================
# COMBINE AND SAVE UK NOMINAL GILT SPOT CURVE
# ============================================================

# Concatenate the two files
uk_gilts_nominal = pd.concat([file_2016, file_2025])

# Sort by date just in case
uk_gilts_nominal = uk_gilts_nominal.sort_index()

# Remove any duplicate dates at the join point
uk_gilts_nominal = uk_gilts_nominal[~uk_gilts_nominal.index.duplicated(keep='first')]

# Final sense check
print("UK Nominal Gilt Spot Curve — combined:")
print(f"  Shape: {uk_gilts_nominal.shape}")
print(f"  Date range: {uk_gilts_nominal.index[0].date()} to {uk_gilts_nominal.index[-1].date()}")
print(f"  Nulls: {uk_gilts_nominal.isnull().sum().sum()}")
print(f"\nFirst 3 rows:")
print(uk_gilts_nominal.head(3))
print(f"\nLast 3 rows:")
print(uk_gilts_nominal.tail(3))

# Save to processed
uk_gilts_nominal.to_csv('data/processed/uk_gilts_nominal.csv')
print("\nSaved to data/processed/uk_gilts_nominal.csv")

UK Nominal Gilt Spot Curve — combined:
  Shape: (2589, 4)
  Date range: 2016-01-04 to 2026-03-31
  Nulls: 0

First 3 rows:
              uk_2yr    uk_5yr   uk_10yr   uk_30yr
0                                                 
2016-01-04  0.611093  1.297485  1.934177  2.684496
2016-01-05  0.593339  1.282928  1.929885  2.679990
2016-01-06  0.534982  1.210266  1.844250  2.614465

Last 3 rows:
              uk_2yr    uk_5yr   uk_10yr   uk_30yr
0                                                 
2026-03-27  4.357465  4.481704  5.009783  5.692511
2026-03-30  4.317637  4.431862  4.957855  5.644072
2026-03-31  4.282974  4.407575  4.944189  5.633331

Saved to data/processed/uk_gilts_nominal.csv


In [17]:
print("Files in data/raw/:")
for f in sorted(os.listdir('data/raw/')):
    print(f"  {f}")

Files in data/raw/:
  GLC Inflation daily data_2016 to 2024.xlsx
  GLC Inflation daily data_2025 to present.xlsx
  GLC Nominal daily data_2016 to 2024.xlsx
  GLC Nominal daily data_2025 to present.xlsx
  GLC Real daily data_2016 to 2024.xlsx
  GLC Real daily data_2025 to present.xlsx


In [18]:
for filepath in [
    'data/raw/GLC Real daily data_2016 to 2024.xlsx',
    'data/raw/GLC Inflation daily data_2016 to 2024.xlsx',
]:
    wb = openpyxl.load_workbook(filepath, read_only=True)
    print(f"{filepath}:")
    for name in wb.sheetnames:
        print(f"  '{name}'")
    wb.close()

data/raw/GLC Real daily data_2016 to 2024.xlsx:
  'info'
  '1. fwds, short end'
  '2. fwd curve'
  '3. spot, short end'
  '4. spot curve'
data/raw/GLC Inflation daily data_2016 to 2024.xlsx:
  'info'
  '1. fwds, short end'
  '2. fwd curve'
  '3. spot, short end'
  '4. spot curve'


In [19]:
# ============================================================
# BOE REAL AND INFLATION SPOT CURVES — IMPORT AND COMBINE
# ============================================================

# --- REAL GILT SPOT CURVE ---
print("Parsing real gilt spot curve...")
real_2016 = parse_boe_spot_curve('data/raw/GLC Real daily data_2016 to 2024.xlsx',
                                  sheet_name='4. spot curve')
real_2025 = parse_boe_spot_curve('data/raw/GLC Real daily data_2025 to present.xlsx',
                                  sheet_name='4. spot curve')

uk_gilts_real = pd.concat([real_2016, real_2025]).sort_index()
uk_gilts_real = uk_gilts_real[~uk_gilts_real.index.duplicated(keep='first')]

# Rename columns to make clear these are real yields
uk_gilts_real.columns = ['uk_real_2yr', 'uk_real_5yr', 'uk_real_10yr', 'uk_real_30yr']

print(f"\nReal gilt spot curve:")
print(f"  Shape: {uk_gilts_real.shape}")
print(f"  Date range: {uk_gilts_real.index[0].date()} to {uk_gilts_real.index[-1].date()}")
print(f"  Nulls: {uk_gilts_real.isnull().sum().sum()}")
print(f"\nFirst 3 rows:")
print(uk_gilts_real.head(3))
print(f"\nLast 3 rows:")
print(uk_gilts_real.tail(3))

# --- INFLATION CURVE ---
print("\nParsing inflation curve...")
inflation_2016 = parse_boe_spot_curve('data/raw/GLC Inflation daily data_2016 to 2024.xlsx',
                                       sheet_name='4. spot curve')
inflation_2025 = parse_boe_spot_curve('data/raw/GLC Inflation daily data_2025 to present.xlsx',
                                       sheet_name='4. spot curve')

uk_inflation_curve = pd.concat([inflation_2016, inflation_2025]).sort_index()
uk_inflation_curve = uk_inflation_curve[~uk_inflation_curve.index.duplicated(keep='first')]

# Rename columns clearly
uk_inflation_curve.columns = ['uk_inf_2yr', 'uk_inf_5yr', 'uk_inf_10yr', 'uk_inf_30yr']

print(f"\nInflation curve:")
print(f"  Shape: {uk_inflation_curve.shape}")
print(f"  Date range: {uk_inflation_curve.index[0].date()} to {uk_inflation_curve.index[-1].date()}")
print(f"  Nulls: {uk_inflation_curve.isnull().sum().sum()}")
print(f"\nFirst 3 rows:")
print(uk_inflation_curve.head(3))
print(f"\nLast 3 rows:")
print(uk_inflation_curve.tail(3))

# --- SAVE BOTH ---
uk_gilts_real.to_csv('data/processed/uk_gilts_real.csv')
uk_inflation_curve.to_csv('data/processed/uk_inflation_curve.csv')
print("\nSaved:")
print("  data/processed/uk_gilts_real.csv")
print("  data/processed/uk_inflation_curve.csv")

Parsing real gilt spot curve...
  Target 2.0yr — found column 2.5yr at index 1
  Target 5.0yr — found column 5.0yr at index 6
  Target 10.0yr — found column 10.0yr at index 16
  Target 30.0yr — found column 30.0yr at index 56
  Target 2.0yr — found column 2.5yr at index 1
  Target 5.0yr — found column 5.0yr at index 6
  Target 10.0yr — found column 10.0yr at index 16
  Target 30.0yr — found column 30.0yr at index 56

Real gilt spot curve:
  Shape: (2589, 4)
  Date range: 2016-01-04 to 2026-03-31
  Nulls: 559

First 3 rows:
            uk_real_2yr  uk_real_5yr  uk_real_10yr  uk_real_30yr
0                                                               
2016-01-04    -1.333642    -0.996678     -0.723996     -0.732845
2016-01-05    -1.362423    -1.021130     -0.740295     -0.737799
2016-01-06    -1.390041    -1.072923     -0.797795     -0.772629

Last 3 rows:
            uk_real_2yr  uk_real_5yr  uk_real_10yr  uk_real_30yr
0                                                               
20

In [20]:
# ============================================================
# INVESTIGATE NULLS AND FIX COLUMN NAMING
# ============================================================

# Where are the nulls concentrated?
print("Nulls per column — real gilts:")
print(uk_gilts_real.isnull().sum())

print("\nNulls per column — inflation curve:")
print(uk_inflation_curve.isnull().sum())

# Are nulls clustered in a specific period?
print("\nReal gilts — null rows by year:")
null_rows = uk_gilts_real[uk_gilts_real.isnull().any(axis=1)]
print(null_rows.groupby(null_rows.index.year).size())

# Show a sample of null rows
print("\nSample of null rows:")
print(null_rows.head(10))

Nulls per column — real gilts:
uk_real_2yr     559
uk_real_5yr       0
uk_real_10yr      0
uk_real_30yr      0
dtype: int64

Nulls per column — inflation curve:
uk_inf_2yr     559
uk_inf_5yr       0
uk_inf_10yr      0
uk_inf_30yr      0
dtype: int64

Real gilts — null rows by year:
0
2016    113
2017     19
2018      9
2019    253
2020     36
2023     65
2024     29
2025     35
dtype: int64

Sample of null rows:
            uk_real_2yr  uk_real_5yr  uk_real_10yr  uk_real_30yr
0                                                               
2016-07-22          NaN    -2.042629     -1.583825     -1.291835
2016-07-25          NaN    -2.052037     -1.584275     -1.291884
2016-07-26          NaN    -2.100108     -1.599807     -1.238566
2016-07-27          NaN    -2.160621     -1.687178     -1.302695
2016-07-28          NaN    -2.155036     -1.693785     -1.320581
2016-07-29          NaN    -2.152512     -1.715488     -1.377562
2016-08-01          NaN    -2.120035     -1.687326     -1.346069

In [21]:
# ============================================================
# FIX COLUMN NAMING AND DROP 2.5YR FROM REAL AND INFLATION
# ============================================================

# Drop the 2.5yr column from real gilts — unreliable and not needed
uk_gilts_real = uk_gilts_real.drop(columns=['uk_real_2yr'])

# Drop the 2.5yr column from inflation curve — same reason
uk_inflation_curve = uk_inflation_curve.drop(columns=['uk_inf_2yr'])

# Sense check
print("Real gilt spot curve — cleaned:")
print(f"  Shape: {uk_gilts_real.shape}")
print(f"  Columns: {list(uk_gilts_real.columns)}")
print(f"  Nulls: {uk_gilts_real.isnull().sum().sum()}")
print(f"\nFirst 3 rows:")
print(uk_gilts_real.head(3))

print("\nInflation curve — cleaned:")
print(f"  Shape: {uk_inflation_curve.shape}")
print(f"  Columns: {list(uk_inflation_curve.columns)}")
print(f"  Nulls: {uk_inflation_curve.isnull().sum().sum()}")
print(f"\nFirst 3 rows:")
print(uk_inflation_curve.head(3))

# Resave cleaned versions
uk_gilts_real.to_csv('data/processed/uk_gilts_real.csv')
uk_inflation_curve.to_csv('data/processed/uk_inflation_curve.csv')
print("\nResaved cleaned files:")
print("  data/processed/uk_gilts_real.csv")
print("  data/processed/uk_inflation_curve.csv")

Real gilt spot curve — cleaned:
  Shape: (2589, 3)
  Columns: ['uk_real_5yr', 'uk_real_10yr', 'uk_real_30yr']
  Nulls: 0

First 3 rows:
            uk_real_5yr  uk_real_10yr  uk_real_30yr
0                                                  
2016-01-04    -0.996678     -0.723996     -0.732845
2016-01-05    -1.021130     -0.740295     -0.737799
2016-01-06    -1.072923     -0.797795     -0.772629

Inflation curve — cleaned:
  Shape: (2589, 3)
  Columns: ['uk_inf_5yr', 'uk_inf_10yr', 'uk_inf_30yr']
  Nulls: 0

First 3 rows:
            uk_inf_5yr  uk_inf_10yr  uk_inf_30yr
0                                               
2016-01-04    2.294164     2.658172     3.417341
2016-01-05    2.304058     2.670180     3.417790
2016-01-06    2.283189     2.642046     3.387094

Resaved cleaned files:
  data/processed/uk_gilts_real.csv
  data/processed/uk_inflation_curve.csv


In [22]:
# ============================================================
# SAVE ALL FRED AND YFINANCE DATA TO PROCESSED
# ============================================================

# US Treasuries
us_treasuries.to_csv('data/processed/us_treasuries.csv')

# US TIPS and inflation expectations
us_tips.to_csv('data/processed/us_tips.csv')

# German Bund
german_bund.to_csv('data/processed/german_bund.csv')

# Policy rates — save separately as different frequencies
boe_rate.to_csv('data/processed/boe_rate.csv', header=True)
fed_funds.to_csv('data/processed/fed_funds.csv', header=True)
ecb_rate.to_csv('data/processed/ecb_rate.csv', header=True)

# Macro variables
macro_variables.to_csv('data/processed/macro_variables.csv')

# yfinance ETFs — combine into one DataFrame
etf_data = pd.DataFrame(yf_data)
etf_data.to_csv('data/processed/etf_data.csv')

# Confirm all files saved
print("All files saved to data/processed/:")
for f in sorted(os.listdir('data/processed/')):
    size = os.path.getsize(f'data/processed/{f}')
    print(f"  {f} ({size/1024:.1f} KB)")

ValueError: If using all scalar values, you must pass an index

In [23]:
# Fix ETF data — concatenate Series explicitly
etf_data = pd.concat(yf_data, axis=1)
etf_data.columns = list(yf_data.keys())
etf_data.to_csv('data/processed/etf_data.csv')

# Now save everything and confirm
print("All files saved to data/processed/:")
for f in sorted(os.listdir('data/processed/')):
    size = os.path.getsize(f'data/processed/{f}')
    print(f"  {f} ({size/1024:.1f} KB)")

All files saved to data/processed/:
  boe_rate.csv (2.2 KB)
  ecb_rate.csv (61.2 KB)
  etf_data.csv (265.8 KB)
  fed_funds.csv (2.0 KB)
  german_bund.csv (3.3 KB)
  macro_variables.csv (63.7 KB)
  uk_gilts_nominal.csv (220.4 KB)
  uk_gilts_real.csv (177.7 KB)
  uk_inflation_curve.csv (170.5 KB)
  us_tips.csv (82.3 KB)
  us_treasuries.csv (80.9 KB)


## Summary

All raw data successfully collected and saved to `/data/processed/`.

### Data collected
- **UK nominal gilt spot curve** — 2yr, 5yr, 10yr, 30yr daily 2016–2026 (BoE portal)
- **UK real gilt spot curve** — 5yr, 10yr, 30yr daily 2016–2026 (BoE portal)
- **UK inflation curve** — 5yr, 10yr, 30yr daily 2016–2026 (BoE portal)
- **US Treasury yields** — 2yr, 5yr, 10yr, 30yr daily 2016–2026 (FRED)
- **US TIPS real yields and breakeven inflation** — 5yr, 10yr daily 2016–2026 (FRED)
- **German Bund 10yr** — monthly 2016–2026 (FRED)
- **Policy rates** — BoE proxy, Fed Funds, ECB monthly 2016–2026 (FRED)
- **Macro variables** — VIX, GBPUSD daily 2016–2026 (FRED)
- **ETF price series** — IGLT.L, INXG.L, TLT, GLD, GBPUSD=X daily 2016–2026 (yfinance)

### Data decisions noted
- UK inflation swap rates unavailable via free sources — breakevens used instead
- BoE real and inflation curves start at 2.5yr minimum — 2yr real yield not available
- Breakeven inflation calculated at 5yr, 10yr, 30yr only where clean maturity matches exist
- BoE rate proxy is OECD 3-month interbank rate (IRSTCI01GBM156N) — tracks Bank Rate closely